# CORNEAL-TRUST: Full Cloud Training (Google Colab T4)

Trains the Phase 2 segmentation U-Net and the Phase 3 severity classifier
at full resolution (384x384) on a free T4 GPU.

### Before starting (one-time, ~5 min)
1. In Google Drive, create a folder named `CornealTrust`.
2. Upload the 4 dataset folders into it, keeping exact names:
   `CORN-1`, `CORN-2`, `CORN-3`, `CORN1500`.
3. This notebook clones the repo from GitHub (must be public).

Then run every cell top-to-bottom (Shift+Enter).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Setup: clone repo + copy datasets to local disk

We **copy** the datasets from Drive into the Colab VM once (about one
minute) instead of symlinking, so training reads from the fast local
disk instead of the slow Drive mount.

In [ ]:
import os, sys, shutil, subprocess

# --- EDIT THIS ---
DATA_PARENT = '/content/drive/MyDrive/CornealTrust'  # folder holding CORN-1/.../CORN1500
REPO = '/content/CORNEAL_TRUST'
# -----------------

if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/gsahoo211004/CORNEAL-TRUST.git', '/content/CORNEAL_TRUST'], check=True)
sys.path.insert(0, REPO)
for name in ['CORN-1', 'CORN-2', 'CORN-3', 'CORN1500']:
    src = os.path.join(DATA_PARENT, name)
    dst = os.path.join('/content', name)
    if os.path.exists(dst):
        print('already present:', name)
    elif os.path.exists(src):
        print('copying', name, '...')
        shutil.copytree(src, dst)
    else:
        print('!!! NOT FOUND in Drive:', src)

In [ ]:
%pip install -q -r "$REPO/requirements.txt" 2>&1 | tail -3
# imagecodecs decodes the LZW/PackBits TIFFs (safety net if above missed it)
%pip install -q imagecodecs 2>&1 | tail -2
print('deps installed')

In [ ]:
%cd "$REPO"
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 1. Segmentation U-Net (Phase 2) -- CORN-1
50-epoch run, early stopping on validation Dice. `--num-workers 4` keeps the
T4 fed. Takes ~20-40 min.

In [ ]:
!python scripts/train_segmentation.py --num-workers 4

import glob
print(glob.glob('outputs/checkpoints/unet_corn1.pt'))

## 2. Severity classifier (Phase 3) -- CORN1500 + CORN-3 val
50-epoch run, early stopping on validation accuracy. Takes ~15-30 min.

In [ ]:
!python scripts/train_severity.py --num-workers 4

import glob
print(glob.glob('outputs/checkpoints/severity_corn1500.pt'))

In [ ]:
# Safety copy: mirror checkpoints + logs back to Drive
out = os.path.join(DATA_PARENT, 'CORNEAL_TRUST_outputs')
shutil.copytree('outputs', os.path.join(out, 'outputs'), dirs_exist_ok=True)
print('saved to', out)
print('IMPORTANT: download unet_corn1.pt and severity_corn1500.pt from there.')